In [ ]:
import os


# Expose only the first T4 GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# import torch

# print("CUDA available:", torch.cuda.is_available())
# print("GPU count:", torch.cuda.device_count())

# if torch.cuda.is_available():
#     print("GPU name:", torch.cuda.get_device_name(0))

#     x = torch.randn(1000, 1000, device="cuda")
#     print("GPU calculation successful:", x.device)

In [3]:
!pip install -q transformers accelerate sentencepiece soundfile librosa

In [4]:
# pip install faster-whisper

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face authentication successful")

Hugging Face authentication successful


In [6]:
import os

VIDEO_PATH = "/kaggle/input/datasets/vidyall/ai-malayalam/AI Workflow Automation Explained_ Why Every Professional Should Learn It _ Malayalam Podcast.mp4"

In [7]:
import subprocess
import os

OUTPUT_DIR = "/kaggle/working/transcription"
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUDIO_PATH = os.path.join(OUTPUT_DIR, "podcast_audio.wav")

command = [
    "ffmpeg",
    "-y",
    "-i", VIDEO_PATH,
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    AUDIO_PATH
]

subprocess.run(command, check=True)

print("Audio extracted:", AUDIO_PATH)
print("File size:", round(os.path.getsize(AUDIO_PATH) / (1024 * 1024), 2), "MB")

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Audio extracted: /kaggle/working/transcription/podcast_audio.wav
File size: 55.53 MB


In [8]:
import sys
import torch
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
else:
    device = "cpu"
    compute_type = "int8"

print("Device:", device)
print("Compute type:", compute_type)


from huggingface_hub import snapshot_download

MODEL_NAME = "bodhan-ai/indic-transcribe-core"

model_dir = snapshot_download(
    repo_id=MODEL_NAME,
    token=hf_token
)

sys.path.insert(0, model_dir)

from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(model_dir)

print("Model loaded successfully")
print("CUDA available:", torch.cuda.is_available())

Device: cuda
Compute type: float16


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1923 [00:00<?, ?it/s]

Model loaded successfully
CUDA available: True


In [9]:
# !pip uninstall -y av
# !pip install "av==18.1.0"

In [10]:
# !pip install faster_whisper

In [11]:
# import av
# import faster_whisper

# print("PyAV version:", av.__version__)
# print("Faster Whisper imported successfully")

In [12]:
# !pip install nvidia-cublas-cu12 nvidia-cudnn-cu12==9.*

In [13]:
# import os
# import site
# import glob

# site_paths = site.getsitepackages()

# cublas_files = []
# cudnn_files = []

# for path in site_paths:
#     cublas_files.extend(
#         glob.glob(path + "/nvidia/cublas/lib/libcublas.so*")
#     )
#     cudnn_files.extend(
#         glob.glob(path + "/nvidia/cudnn/lib/libcudnn.so*")
#     )

# print("cuBLAS files:", cublas_files[:5])
# print("cuDNN files:", cudnn_files[:5])

In [14]:
# import os
# import ctypes

# base = "/usr/local/lib/python3.13/dist-packages/nvidia"

# cublas_path = base + "/cublas/lib"
# cudnn_path = base + "/cudnn/lib"

# os.environ["LD_LIBRARY_PATH"] = (
#     cublas_path + ":" + cudnn_path + ":" +
#     os.environ.get("LD_LIBRARY_PATH", "")
# )

# # Load libraries explicitly into this Python process
# ctypes.CDLL(cublas_path + "/libcublas.so.12")
# ctypes.CDLL(cudnn_path + "/libcudnn.so.9")

# print("cuBLAS loaded!")
# print("cuDNN loaded!")

In [15]:
import librosa
import soundfile as sf

audio, sr = librosa.load(
    AUDIO_PATH,
    sr=16000,
    mono=True
)

# Extract first 30 seconds
sample = audio[:30 * 16000]

sf.write(
    "/kaggle/working/sample.wav",
    sample,
    16000
)

In [16]:
# from long_form import transcribe_long

# transcript = transcribe_long(
#     asr,
#     "/kaggle/working/sample.wav",
#     lang="ml",
#     return_lid=True
# )

# # print("Language information:", lid)
# print("\nTranscription:")
# print(transcript)

In [17]:
import librosa
import soundfile as sf
import os

audio, sr = librosa.load(
    AUDIO_PATH,
    sr=16000,
    mono=True
)

chunk_duration = 25
chunk_samples = chunk_duration * sr

transcript_chunks = []

os.makedirs("/kaggle/working/chunks", exist_ok=True)

for i, start in enumerate(range(0, len(audio), chunk_samples)):

    end = min(start + chunk_samples, len(audio))

    chunk = audio[start:end]

    chunk_path = f"/kaggle/working/chunks/chunk_{i}.wav"

    sf.write(chunk_path, chunk, sr)

    text, lid = asr.transcribe(
        chunk_path,
        lang="ml",
        return_lid=True
    )

    transcript_chunks.append({
        "start": round(start / sr, 2),
        "end": round(end / sr, 2),
        "text": text
    })

    print(f"Processed chunk {i + 1}")

Processed chunk 1
Processed chunk 2
Processed chunk 3
Processed chunk 4
Processed chunk 5
Processed chunk 6
Processed chunk 7
Processed chunk 8
Processed chunk 9
Processed chunk 10
Processed chunk 11
Processed chunk 12
Processed chunk 13
Processed chunk 14
Processed chunk 15
Processed chunk 16
Processed chunk 17
Processed chunk 18
Processed chunk 19
Processed chunk 20
Processed chunk 21
Processed chunk 22
Processed chunk 23
Processed chunk 24
Processed chunk 25
Processed chunk 26
Processed chunk 27
Processed chunk 28
Processed chunk 29
Processed chunk 30
Processed chunk 31
Processed chunk 32
Processed chunk 33
Processed chunk 34
Processed chunk 35
Processed chunk 36
Processed chunk 37
Processed chunk 38
Processed chunk 39
Processed chunk 40
Processed chunk 41
Processed chunk 42
Processed chunk 43
Processed chunk 44
Processed chunk 45
Processed chunk 46
Processed chunk 47
Processed chunk 48
Processed chunk 49
Processed chunk 50
Processed chunk 51
Processed chunk 52
Processed chunk 53
Pr

In [18]:
print(transcript_chunks)

[{'start': 0.0, 'end': 25.0, 'text': 'അപ്പോൾ ഈ ഓട്ടോമേഷനും എ ഐ വർക്ക് ഫ്ലോ ഓട്ടോമേഷനും തമ്മിലുള്ള വ്യത്യാസം എ തന്നെ ഇപ്പം വരുന്ന ഓരോ മെസ്സേജിനും എന്ത് റിപ്ലൈ കൊടുക്കണമെന്ന് എ തന്നെ കസ്റ്റമൈസ് ചെയ്ത് നമ്മൾ അതിന് ആ ഒരു പർട്ടിക്കുലർ റീസണിങ്ങൊക്കെ വെച്ച് സെറ്റാക്കി വെച്ചിട്ടുണ്ട് പഠിക്കണം എന്ന് എനിക്ക് ആഗ്രഹമുണ്ട് ഞാൻ പെയ്ഡ് എടുത്തിട്ട് ഇപ്പോൾ അത് മുന്നോട്ട് പോയാൽ അതൊരു ഫീസിബിൾ സൊല്യൂഷൻ ആണോ എന്നിട്ട് നമ്മുടെ പ്ലാറ്റ്ഫോം ഒരു ഓപ്പൺ സോഴ്സ് പ്ലാറ്റ്ഫോം ആണ് നമുക്ക് വേണമെങ്കിൽ നമ്മുടെ എൻഡിൽ തന്നെ സെൽഫ് ഹോസ്റ്റ് ചെയ്യേണ്ട ഓപ്ഷൻ ഉണ്ട് പക്ഷേ ഇത്തിരി'}, {'start': 25.0, 'end': 50.0, 'text': 'ടെക്നിക്കലാണ് അല്ലാതെ നമുക്ക് എൻ ഐ ടിയിൽ നിന്ന് തന്നെ ഒരു ഫോർട്ടീൻ ഡേയ്സ് ഫ്രീ ട്രയൽ ആയിരുന്നു അത് ഇങ്ങനെ ഒരു മിനിറ്റിലുമാണ് പുതിയ പുതിയ എന്തെല്ലോ വരുന്നത് വിശ്വാസമാണെങ്കിൽ അപ്പോൾ നമ്മൾ ഇങ്ങനെ ഒരു സാധനം പഠിച്ചു കഴിഞ്ഞാൽ നാളെ ഇത് ഉണ്ടാകുമോ എന്നുള്ളൊരു ചോദ്യം ഇപ്പം പലർക്കും ഉണ്ട് നമ്മൾ ഒരിക്കലും ഒരു ടൂളിനെ മാത്രം ഫോക്കസ് ചെയ്യല്ലേ പഠിപ്പിക്കുന്നത് നമ്മളൊരു ഓട്ടോമേഷൻ എങ്ങനെ സ്ട്രക്ച്ചർ ചെയ്യാം ഒരു വർക്ക് ഫ്ലോ എങ്ങനെ

In [19]:
!pip install -q transformers

In [20]:
# from transformers import pipeline

# classifier = pipeline(
#     "zero-shot-classification",
#     model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
# )

In [21]:
!pip install -q wtpsplit

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [22]:
from wtpsplit import SaT

sat = SaT("sat-3l-sm")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  855MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

In [23]:
def clean_segments(sentences, min_words=4):

    merged = []
    buffer = ""

    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue

        if buffer:
            sentence = buffer + " " + sentence
            buffer = ""

        if len(sentence.split()) < min_words:
            buffer = sentence
        else:
            merged.append(sentence)

    if buffer:
        if merged:
            merged[-1] += " " + buffer
        else:
            merged.append(buffer)

    return merged

In [24]:
cleaned_sentences = []

for chunk in transcript_chunks:

    sentences = sat.split(chunk["text"])

    merged = clean_segments(
        sentences,
        min_words=4
    )

    for sentence in merged:

        cleaned_sentences.append({
            "chunk_start": chunk["start"],
            "chunk_end": chunk["end"],
            "text": sentence
        })

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [25]:
!pip install -q thx01

/usr/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=51) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [26]:
import thx01

agent = thx01.load("LatentAILabs/yama-indic")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [27]:
def classify_sentences(texts, batch_size=16):

    results = []

    for i in range(0, len(texts), batch_size):

        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=-1)

        predicted_ids = torch.argmax(
            probabilities, dim=-1
        )

        for j in range(len(batch_texts)):

            label_id = predicted_ids[j].item()

            confidence = probabilities[j, label_id].item()

            results.append({
                "label": model.config.id2label[label_id],
                "confidence": round(confidence, 3)
            })

    return results

In [28]:
sentence_texts = [
    item["text"] for item in cleaned_sentences
]

from tqdm.auto import tqdm

results = []
questions_v2 = {
    "classification": {
        "type": "choice",
        "instructions": """
        Classify the conversational function of this Malayalam
        podcast transcript sentence.

        Classify based on actual conversational meaning.
        A sentence mentioning a question is not necessarily a question.
        A sentence containing how or what is not always a question.
        """,
        "criteria": {
            "question": """
            The speaker directly asks another person a question,
            requests information, or seeks an answer.
            """,
            "agreement": """
            The speaker explicitly agrees with, confirms,
            or supports a previous speaker's opinion.
            """,
            "disagreement": """
            The speaker explicitly rejects, challenges,
            corrects, or opposes a previous speaker's opinion.
            """,
            "other": """
            General statements, explanations, descriptions,
            examples, suggestions, observations, and narration.
            """
        }
    }
}
for text in tqdm(sentence_texts):

    try:
        result = agent.system_one(text, questions_v2)

        classification = result["answers"]["classification"]

        results.append({
            "prediction": classification["choice"],
            "confidence": classification["confidence"],
            "question_prob": classification["probabilities"]["question"],
            "agreement_prob": classification["probabilities"]["agreement"],
            "disagreement_prob": classification["probabilities"]["disagreement"],
            "other_prob": classification["probabilities"]["other"]
        })

    except Exception as e:
        print("Error:", e)

        results.append({
            "prediction": "error",
            "confidence": 0,
            "question_prob": 0,
            "agreement_prob": 0,
            "disagreement_prob": 0,
            "other_prob": 0
        })

  0%|          | 0/511 [00:00<?, ?it/s]

In [29]:
import pandas as pd

classified_df = pd.DataFrame([
    {
        "chunk_start": item["chunk_start"],
        "chunk_end": item["chunk_end"],
        "text": item["text"],
        **result
    }
    for item, result in zip(cleaned_sentences, results)
])

display(classified_df)

,chunk_start,chunk_end,text,prediction,confidence,question_prob,agreement_prob,disagreement_prob,other_prob
0,0.0,25.00,അപ്പോൾ ഈ ഓട്ടോമേഷനും എ ഐ വർക്ക് ഫ്ലോ ഓട്ടോമേഷന...,question,0.4581,0.6866,0.0167,0.0167,0.2800
1,0.0,25.00,പഠിക്കണം എന്ന് എനിക്ക് ആഗ്രഹമുണ്ട്,other,0.3737,0.4422,0.0243,0.0189,0.5146
2,0.0,25.00,ഞാൻ പെയ്ഡ് എടുത്തിട്ട് ഇപ്പോൾ അത് മുന്നോട്ട് പ...,question,0.8196,0.9508,0.0155,0.0154,0.0184
3,0.0,25.00,എന്നിട്ട് നമ്മുടെ പ്ലാറ്റ്ഫോം ഒരു ഓപ്പൺ സോഴ്സ്...,other,0.7825,0.0187,0.0276,0.0164,0.9373
4,0.0,25.00,നമുക്ക് വേണമെങ്കിൽ നമ്മുടെ എൻഡിൽ തന്നെ സെൽഫ് ഹ...,other,0.6421,0.0391,0.0537,0.0277,0.8795
...,...,...,...,...,...,...,...,...,...
506,1775.0,1800.00,അടുത്ത ഒരു ഇൻഫോർമേറ്റീവായിട്ടുള്ള പോഡ്കാസ്റ്റ്...,other,0.4944,0.1331,0.0415,0.0342,0.7913
507,1800.0,1819.65,അതുവരെ നമുക്ക് വെയിറ്റ് ചെയ്യാം,other,0.4289,0.0979,0.0835,0.0517,0.7668
508,1800.0,1819.65,മാറ്റം ഉടനെ ഉണ്ടാവും മാറ്റം ഉടനെ ഉണ്ടാവും,other,0.2667,0.1900,0.1075,0.0635,0.6390
509,1800.0,1819.65,എല്ലാവരും മാറണം എല്ലാവരും അപ്ഗ്രേഡ് ചെയ്യണം,other,0.1982,0.0576,0.3730,0.0962,0.4732


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
highlights_df = classified_df[
    (classified_df["prediction"].isin([
        "question",
        "agreement",
        "disagreement"
    ])) &
    (classified_df["confidence"] >= 0.50)
].copy()

display(highlights_df)

,chunk_start,chunk_end,text,prediction,confidence,question_prob,agreement_prob,disagreement_prob,other_prob
2,0.0,25.0,ഞാൻ പെയ്ഡ് എടുത്തിട്ട് ഇപ്പോൾ അത് മുന്നോട്ട് പ...,question,0.8196,0.9508,0.0155,0.0154,0.0184
8,25.0,50.0,പുതിയ പുതിയ എന്തെല്ലോ വരുന്നത്,question,0.7901,0.9400,0.0165,0.0162,0.0273
9,25.0,50.0,വിശ്വാസമാണെങ്കിൽ അപ്പോൾ നമ്മൾ ഇങ്ങനെ ഒരു സാധനം...,question,0.8340,0.9557,0.0145,0.0143,0.0156
19,50.0,75.0,ഹായ് ഹലോ ഹായ് അഭി ഹലോ ഹായ് വിഷ്ണു,question,0.6871,0.8973,0.0255,0.0210,0.0561
22,75.0,100.0,ഓക്കെ മാറ്റത്തിൻ്റെ ലാസ്റ്റ് പോഡ്കാസ്റ്റ് ഒക്ക...,question,0.8180,0.9502,0.0155,0.0153,0.0190
...,...,...,...,...,...,...,...,...,...
495,1725.0,1750.0,ട്രൂ കറക്റ്റ് ആണ് അല്ലേ പഴയ ക്ലൈൻറ്റ്സ്,question,0.7741,0.9346,0.0225,0.0213,0.0215
501,1750.0,1775.0,യെസ് ഓക്കെ അപ്പോൾ വേറെ എന്തെങ്കിലും അഡീഷണൽ ഡേയ...,question,0.8326,0.9552,0.0150,0.0139,0.0158
502,1750.0,1775.0,ഉണ്ടോ വേറെ എന്തെങ്കിലും ഇല്ല വെൽ,question,0.8185,0.9503,0.0149,0.0148,0.0200
503,1775.0,1800.0,അപ്പോൾ എല്ലാവരും ഈ ഒരു വർക്ക് ഫ്ലോ ഓട്ടോമേഷൻ പ...,question,0.7207,0.9133,0.0214,0.0258,0.0395


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [31]:
print(classified_df["prediction"].value_counts())

prediction
other           334
question        135
agreement        35
disagreement      7
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# def merge_ranges(df):
#     ranges = sorted(
#         zip(df["chunk_start"], df["chunk_end"])
#     )

#     merged = []

#     for start, end in ranges:
#         if not merged or start > merged[-1][1]:
#             merged.append([start, end])
#         else:
#             merged[-1][1] = max(merged[-1][1], end)

#     return merged

# merged_ranges = merge_ranges(classified_df)

# print(merged_ranges)

In [33]:
output_dir = "/kaggle/working/highlight_clips"

In [34]:
MIN_DURATION = 30
MAX_DURATION = 60
CLIP_DURATION = 45

candidates = []

for _, row in highlights_df.iterrows():

    highlight_time = (row["chunk_start"] + row["chunk_end"]) / 2

    start = max(0, highlight_time - CLIP_DURATION / 2)
    end = start + CLIP_DURATION

    candidates.append({
        "start": start,
        "end": end,
        "category": row["prediction"],
        "text": row["text"]
    })

print("Candidate clips:", len(candidates))

Candidate clips: 104


In [35]:
candidates = sorted(candidates, key=lambda x: x["start"])

merged_candidates = []

for clip in candidates:

    if not merged_candidates:
        merged_candidates.append(clip)
        continue

    previous = merged_candidates[-1]

    if clip["start"] < previous["end"]:
        previous["end"] = max(previous["end"], clip["end"])
        previous["text"] += " " + clip["text"]

        if clip["category"] not in previous["category"]:
            previous["category"] += ", " + clip["category"]

    else:
        merged_candidates.append(clip)

print("Clips after merging:", len(merged_candidates))


Clips after merging: 16


In [ ]:
import os
import subprocess
output_dir = "/kaggle/working/highlight_clips"
os.makedirs(output_dir, exist_ok=True)

for i, clip in enumerate(merged_candidates, start=1):

    start = clip["start"]
    duration = min(clip["end"] - start, MAX_DURATION)

    if duration < MIN_DURATION:
        continue

    output_path = os.path.join(
        output_dir,
        f"highlight_{i:03d}.mp4"
    )

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-ss", str(start),
        "-i", VIDEO_PATH,
        "-t", str(duration),
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-crf", "23",
        "-c:a", "aac",
        "-movflags", "+faststart",
        output_path
    ]

    subprocess.run(cmd, check=True)

    print(f"Saved: {output_path}")

Saved: /kaggle/working/highlight_clips/highlight_001.mp4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved: /kaggle/working/highlight_clips/highlight_002.mp4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved: /kaggle/working/highlight_clips/highlight_003.mp4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


... (10 outputs left out)

Saved: /kaggle/working/highlight_clips/highlight_009.mp4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved: /kaggle/working/highlight_clips/highlight_010.mp4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
    display(Video("/kaggle/working/highlight_clips/highlight_001.mp4", embed=True))